# W7D5_LAB_In_God_we_trust_everyone_else_bring_the_data
- Week 7 / Day 5
- Student: Andreas Papachristophorou
- Course: AI Consulting & Integration 2026-07
- Date: 2026-08-24
---

## Step 1: Find and Select a Dataset


## Dataset Selection

### Dataset Name
Digital Marketing Campaign Dataset

### Source
Kaggle

### Dataset URL
https://www.kaggle.com/datasets/arpit2712/digital-marketing-company/data

### Why This Dataset Was Chosen
This dataset was selected because it contains 8,000 marketing campaign/customer records across five campaign channels: Email, PPC, SEO, Social Media, and Referral. These channels provide multiple comparable groups, which is required for pairwise statistical testing.

The dataset includes useful marketing-performance variables, including advertising spend (`AdSpend`), click-through rate (`ClickThroughRate`), conversion rate (`ConversionRate`), website visits (`WebsiteVisits`), and a binary conversion outcome (`Conversion`). This supports both continuous-metric comparisons using independent t-tests and binary-outcome comparisons using Fisher's exact test.

The dataset does not include revenue, so a genuine Return on Ad Spend (ROAS) calculation is not possible. The analysis will therefore focus on conversion performance, advertising spend, click-through rate, and conversion outcomes. This limitation will be clearly stated in the final executive memo.

Although the dataset is general digital marketing data rather than cruise-industry data, the findings will be interpreted as an illustrative digital customer-acquisition scenario relevant to tourism or cruise marketing. No conclusions will be presented as evidence about a real cruise company.

## Step 2: Load and Explore Your Dataset

In [ ]:
import pandas as pd

file_path = "data/digital_marketing_campaign_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

In [ ]:
# Count how many rows belong to each marketing channel
channel_counts = df["CampaignChannel"].value_counts()

# Print a clear heading before the results
print("Records per marketing channel:")

# Display the count for each channel
print(channel_counts)

### Channel Sample Sizes

The dataset contains five campaign channels with relatively balanced sample sizes:

- Referral: 1,719 records
- PPC: 1,655 records
- Email: 1,557 records
- SEO: 1,550 records
- Social Media: 1,519 records

Each channel has more than 1,500 records. This provides sufficient observations for exploratory analysis and pairwise statistical comparisons. Although the group sizes are not identical, the difference between the largest and smallest group is modest, so no channel is severely under-represented.

In [ ]:
# List the key columns we plan to use in the analysis
key_columns = [
    "CampaignChannel",
    "AdSpend",
    "ClickThroughRate",
    "ConversionRate",
    "WebsiteVisits",
    "Conversion"
]

# Count missing values in each key column
missing_values = df[key_columns].isna().sum()

# Print a clear heading before the result
print("Missing values in key analysis columns:")

# Display the number of missing values for each selected column
print(missing_values)

### Missing-Value Check

No missing values were found in the key analysis columns: `CampaignChannel`, `AdSpend`, `ClickThroughRate`, `ConversionRate`, `WebsiteVisits`, and `Conversion`.

Therefore, no missing-value imputation or row removal is required for these variables. All 8,000 records can remain in the analysis.

In [ ]:
# Display summary statistics for the main numeric columns
print(df[["AdSpend", "ClickThroughRate", "ConversionRate", "WebsiteVisits"]].describe().round(4))

# Show every unique value found in the binary Conversion column
print("\nUnique values in Conversion:")
print(sorted(df["Conversion"].unique()))

In [ ]:
# Group all records by marketing channel
# Calculate the number of records and the average value of each available performance metric
channel_summary = (
    df.groupby("CampaignChannel")
    .agg(
        Records=("CustomerID", "count"),
        Average_AdSpend=("AdSpend", "mean"),
        Average_CTR=("ClickThroughRate", "mean"),
        Average_ConversionRate=("ConversionRate", "mean"),
        Average_WebsiteVisits=("WebsiteVisits", "mean"),
        Binary_Conversion_Rate=("Conversion", "mean")
    )
    .round(4)
    .sort_values("Average_ConversionRate", ascending=False)
)

# Convert decimal rates to percentages so the final table is easier to read
channel_summary["Average_CTR"] = channel_summary["Average_CTR"] * 100
channel_summary["Average_ConversionRate"] = channel_summary["Average_ConversionRate"] * 100
channel_summary["Binary_Conversion_Rate"] = channel_summary["Binary_Conversion_Rate"] * 100

# Round the percentage columns again after multiplying by 100
channel_summary = channel_summary.round(2)

# Display the completed summary table
print("Marketing Channel Performance Summary:")
display(channel_summary)

In [ ]:
%pip install matplotlib seaborn

In [ ]:
# Import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns
# Import Path, which helps Python work safely with folders and file paths
from pathlib import Path

# Sort from lowest to highest average spend
ad_spend_plot = channel_summary.sort_values(
    "Average_AdSpend",
    ascending=True
)

# Create a larger figure
plt.figure(figsize=(10, 5))

# Draw the bars
ax = sns.barplot(
    data=ad_spend_plot,
    x="Average_AdSpend",
    y=ad_spend_plot.index,
    color="steelblue"
)

# Zoom the x-axis because all channel averages are close to $5,000
plt.xlim(4800, 5200)

# Add a title and clear labels
plt.title("Average Advertising Spend by Campaign Channel", fontsize=14, weight="bold")
plt.xlabel("Average Ad Spend (USD)", fontsize=11)
plt.ylabel("Campaign Channel", fontsize=11)

# Add light vertical guide lines
plt.grid(axis="x", linestyle="--", alpha=0.5)

# Add the exact dollar value at the end of every bar
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"${value:,.2f}" for value in ad_spend_plot["Average_AdSpend"]],
        padding=4
    )


# Create the graphs folder if it does not already exist
Path("graphs").mkdir(exist_ok=True)
# Save the chart inside the graphs subfolder
plt.savefig("graphs/average_ad_spend_by_channel.png", dpi=300)

# Show the chart in the notebook
plt.show()

### Initial observation: average ad spend

Average advertising spend is very similar across all five campaign channels, ranging from approximately $4,954 for PPC to $5,056 for Email. The largest difference is about $101, which is small relative to the overall average spend of about $5,000. This suggests that channel performance differences, if present, are unlikely to be explained by large differences in average ad spend alone. Further statistical testing is required before making budget-allocation recommendations.

In [ ]:
# Sort channels from lowest to highest conversion rate
conversion_rate_plot = channel_summary.sort_values(
    "Average_ConversionRate",
    ascending=True
)

# Create the figure
plt.figure(figsize=(10, 5))

# Create a horizontal bar chart
ax = sns.barplot(
    data=conversion_rate_plot,
    x="Average_ConversionRate",
    y=conversion_rate_plot.index,
    color="seagreen"
)

# Add labels and title
plt.title("Average Conversion Rate by Campaign Channel", fontsize=14, weight="bold")
plt.xlabel("Average Conversion Rate (%)", fontsize=11)
plt.ylabel("Campaign Channel", fontsize=11)

# Zoomed axis highlights small differences; it does not imply large performance gaps
plt.xlim(9, 11)

# Add guide lines
plt.grid(axis="x", linestyle="--", alpha=0.5)

# Put the exact percentage beside each bar
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[
            f"{value:.2f}%"
            for value in conversion_rate_plot["Average_ConversionRate"]
        ],
        padding=4
    )

# Save the chart to the graphs folder
plt.tight_layout()
plt.savefig(
    "graphs/average_conversion_rate_by_channel.png",
    dpi=300,
    bbox_inches="tight"
)

# Display the chart
plt.show()

### Initial observation: average conversion rate

Social Media has the highest average conversion rate at 10.66%, followed by Email at 10.49%. Referral has the lowest average conversion rate at 10.31%.

The overall range across channels is only 0.35 percentage points, from 10.31% to 10.66%. The chart uses a truncated x-axis from 9% to 11% to make these small differences visible; therefore, the visual bar-length differences should not be interpreted as large performance gaps.

At this exploratory stage, the chart shows descriptive differences only. Pairwise t-tests, effect sizes, and multiple-comparisons correction are required before concluding that any channel has a statistically or practically meaningful advantage.

In [ ]:
# Create a larger figure for the distribution comparison
plt.figure(figsize=(10, 6))

# Show the spread of individual conversion-rate values in each channel
sns.boxplot(
    data=df,
    x="ConversionRate",
    y="CampaignChannel",
    color="seagreen"
)

# Convert the x-axis from decimals to percentage-style labels
plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(lambda value, position: f"{value * 100:.0f}%")
)

# Add a descriptive title and labels
plt.title("Distribution of Conversion Rates by Campaign Channel", fontsize=14, weight="bold")
plt.xlabel("Conversion Rate", fontsize=11)
plt.ylabel("Campaign Channel", fontsize=11)

# Add light guide lines for comparison
plt.grid(axis="x", linestyle="--", alpha=0.5)

# Save the required distribution visualization
plt.tight_layout()
plt.savefig(
    "graphs/conversion_rate_distributions_by_channel.png",
    dpi=300,
    bbox_inches="tight"
)

# Display the chart
plt.show()